# Pro Plan Churn Drivers (2025) — Debug Notebook

**Analyst:** Nimrod Fisher · **Pulseboard** · Standalone walkthrough of the full analysis.

This notebook reloads the **saved CSVs** in `../results/` (no database connection) and rebuilds the key tables and charts, phase by phase: Plan → QA → EDA → Deep Analysis → Synthesis → Validation → Conclusions.

> Headline: in 2025, Pro "churn" is **subscription downsizing, not customer loss** — 0 of 15 Pro accounts fully left. Cancellations concentrate in the **$199 tier within its first 90 days**; the **$29 tier has never churned**.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

RESULTS = Path('..') / 'results'
QA, EDA, DA, VAL = RESULTS/'qa', RESULTS/'eda', RESULTS/'deep-analysis', RESULTS/'validation'
pd.set_option('display.max_columns', None)
INK, RUST, AMBER, TEAL, MUTED = '#14110f', '#b8431f', '#d98a3d', '#2f5d52', '#cdbfa9'
print('Loading saved CSVs from', RESULTS.resolve())

## Phase 1 — Plan

**Question:** What drives churn (subscription cancellation) among Pro-plan accounts, viewed against the 2025 active Pro base?

**Hypotheses:** H1 price tier · H2 early-life tenure · H3 engagement · H4 support burden · H0 no driver / rationalization.

**Known issues applied:** `plan` is lowercase (`'pro'`); price tier = `subscriptions.monthly_price` (the products table carries no price signal); events cover only 2025-03-07–06-06; n is small (7 cancels).

## Phase 2 — Data QA (score 84/100)
Core churn data is pristine; the issues shape *how* we read it.

In [ ]:
pop = pd.read_csv(QA/'00_qa-pro-population.csv')
comp = pd.read_csv(QA/'00_qa-subscriptions-completeness.csv')
cov = pd.read_csv(QA/'00_qa-event-coverage.csv')
pmap = pd.read_csv(QA/'00_qa-product-mapping.csv')
print('Plan distribution:'); display(pop)
print('Subscription completeness (all zeros = clean):'); display(comp)
print('Event coverage vs cancels:'); display(cov)
print('Product price mismatch — catalog never equals sub price:'); display(pmap)

## Phase 3 — EDA
Cancel rate by tier, tenure split, rationalization.

In [ ]:
tier = pd.read_csv(EDA/'01_eda-cancel-rate-by-tier.csv')
display(tier)
fig, ax = plt.subplots(figsize=(7,4))
colors = [MUTED, AMBER, RUST]
ax.bar(tier['monthly_price'].astype(str)+' /mo', tier['cancel_rate_pct'], color=colors)
for i,(p,r) in enumerate(zip(tier['monthly_price'], tier['cancel_rate_pct'])):
    ax.text(i, r+0.6, f'{r:.1f}%', ha='center', fontweight='bold')
ax.set_title('Cancel rate climbs with price tier'); ax.set_ylabel('cancel rate (%)'); ax.set_ylim(0,40)
plt.tight_layout(); plt.show()

In [ ]:
ten = pd.read_csv(EDA/'02_eda-tenure.csv')
display(ten)
canc = ten[ten['status']=='canceled']
fig, ax = plt.subplots(figsize=(7,3.2))
labels = canc['status']+' $'+canc['monthly_price'].astype(str)
ax.barh(labels, canc['median_tenure'], color=RUST)
ax.axvline(90, color=INK, ls='--', lw=1); ax.text(92, -0.3, '90d', color=INK)
for i,v in enumerate(canc['median_tenure']): ax.text(v+5, i, f'{v:.0f}d', va='center')
ax.set_title('Median tenure of canceled subs by tier'); ax.set_xlabel('days')
plt.tight_layout(); plt.show()
print('$199 cancels are early-life (median 24d); $79 cancels are late (median ~244d).')

In [ ]:
rat = pd.read_csv(EDA/'07_eda-rationalization.csv'); display(rat)
eng = pd.read_csv(EDA/'05_eda-engagement.csv'); print('Engagement (paradox — churners more active):'); display(eng)
sup = pd.read_csv(EDA/'06_eda-support.csv'); print('Support (no separation):'); display(sup)

## Phase 4 — Deep Analysis
Quantify H1 (entry vs premium), H2 (90-day hazard), H0 (rationalization detail).

In [ ]:
rates = pd.read_csv(DA/'08_da-tier-rates.csv'); print('Entry vs premium:'); display(rates)
haz = pd.read_csv(DA/'09_da-199-hazard.csv'); print('$199 first-90-day hazard:'); display(haz)
rd = pd.read_csv(DA/'10_da-rationalization-detail.csv'); print('Account-level: dropped vs kept:'); display(rd)

## Phase 5 — Synthesis (verdicts)
| H | Strength | Answer |
|---|---|---|
| H1 Price | Moderate | $29 0% vs premium 30.4%; real threshold is entry-vs-paid, not a gradient. |
| H2 Tenure | Moderate | $199 early-life (median 24d, 90-day hazard 30.8%); $79 late. |
| H3 Engagement | Refuted | Churners *more* active (47 vs 39). |
| H4 Support | Null | 2.0 vs 1.5 tickets/account. |
| H0 Rationalization | Strong | 4/5 kept an active sub; 0 true logo exits in 2025. |

## Phase 6 — Validation
Churn-definition sensitivity, cohort confound (rejected), account-grain replication.

In [ ]:
defn = pd.read_csv(VAL/'11_val-churn-definition.csv'); print('Churn rate by definition (7.7% – 38.5%):'); display(defn)
conf = pd.read_csv(VAL/'12_val-tier-cohort-confound.csv'); print('Cohort age by tier ($199 not meaningfully newer):'); display(conf)
rep = pd.read_csv(VAL/'13_val-replication-account-tier.csv'); print('Account-grain replication of H1:'); display(rep)

## Conclusions

1. **Pro "churn" in 2025 is downsizing, not loss** — 0 true logo exits; 4/5 cancelling accounts kept an active subscription.
2. **Drivers of cancellation:** price tier ($29 churn-free; paid tiers ~30%) and **early-life $199 tenure** (median 24d, ~31% 90-day hazard).
3. **Non-drivers:** engagement (refuted — churners more active) and support burden (null).
4. **Act on:** value-realization for new $199 subscriptions in the first 90 days; reframe internal churn reporting to separate downsizing from logo churn.

**Caveats:** n=7 (descriptive, not significant); churn is definition-sensitive; durability of "0 exits" beyond the 2025-06-17 snapshot is the key open question.